# Painting with Data

One of the most iconic data visualizations of the past decade is the [climate spiral](https://en.wikipedia.org/wiki/Climate_spiral), first published by climate scientist Ed Hawkins in 2016. It maps monthly global temperature anomalies onto a polar coordinate system, where each year traces a ring and the spiral expands outward as the planet warms.

In this notebook we recreate that idea using the p5.js kernel, importing the [chroma-js](https://gka.github.io/chroma.js/) color library from npm to handle perceptually uniform color mapping. The result is a data-driven generative piece: part chart, part art.

In [ ]:
import chroma from 'chroma-js';

## The Data

We embed monthly global surface temperature anomalies from [NASA GISS](https://data.giss.nasa.gov/gistemp/) (GISTEMP v4), covering 2010–2024. Each value is the deviation in °C from the 1951–1980 baseline. The trend is unmistakable: the spiral drifts outward year after year, with 2023–2024 pushing well past the +1.0 °C mark.

In [ ]:
const startYear = 2010;
const monthNames = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'];

// NASA GISS monthly global temperature anomalies (°C vs 1951-1980 baseline)
const raw = [
  // 2010
  0.75, 0.84, 0.92, 0.85, 0.76, 0.67, 0.63, 0.67, 0.63, 0.71, 0.81, 0.45,
  // 2011
  0.53, 0.48, 0.66, 0.64, 0.53, 0.63, 0.70, 0.75, 0.56, 0.65, 0.59, 0.60,
  // 2012
  0.47, 0.48, 0.57, 0.73, 0.78, 0.65, 0.59, 0.66, 0.72, 0.80, 0.78, 0.53,
  // 2013
  0.71, 0.62, 0.67, 0.53, 0.60, 0.70, 0.60, 0.71, 0.77, 0.69, 0.83, 0.70,
  // 2014
  0.77, 0.55, 0.78, 0.81, 0.87, 0.67, 0.58, 0.84, 0.88, 0.80, 0.67, 0.79,
  // 2015
  0.87, 0.91, 0.96, 0.75, 0.80, 0.81, 0.73, 0.80, 0.84, 1.08, 1.06, 1.17,
  // 2016
  1.18, 1.37, 1.35, 1.09, 0.96, 0.81, 0.85, 1.02, 0.90, 0.87, 0.91, 0.86,
  // 2017
  1.02, 1.14, 1.17, 0.94, 0.91, 0.71, 0.82, 0.88, 0.74, 0.89, 0.87, 0.93,
  // 2018
  0.82, 0.85, 0.88, 0.89, 0.81, 0.77, 0.84, 0.78, 0.80, 1.02, 0.84, 0.93,
  // 2019
  0.93, 0.96, 1.17, 1.01, 0.86, 0.91, 0.94, 0.95, 0.93, 1.00, 0.99, 1.12,
  // 2020
  1.18, 1.24, 1.18, 1.12, 1.00, 0.91, 0.89, 0.86, 0.97, 0.87, 1.08, 0.80,
  // 2021
  0.82, 0.64, 0.89, 0.76, 0.79, 0.85, 0.92, 0.81, 0.93, 0.99, 0.93, 0.88,
  // 2022
  0.91, 0.89, 1.05, 0.84, 0.84, 0.92, 0.94, 0.95, 0.89, 0.97, 0.73, 0.80,
  // 2023
  0.88, 0.97, 1.23, 0.99, 0.94, 1.09, 1.20, 1.19, 1.48, 1.34, 1.41, 1.37,
  // 2024
  1.25, 1.45, 1.39, 1.31, 1.15, 1.21, 1.20, 1.29, 1.24, 1.33, 1.30, 1.26,
];

const data = raw.map((anomaly, i) => ({
  year: startYear + Math.floor(i / 12),
  month: i % 12,
  anomaly,
  index: i,
}));

const endYear = startYear + Math.floor((raw.length - 1) / 12);
const totalYears = endYear - startYear + 1;

const summary = {
  period: startYear + '\u2013' + endYear,
  months: raw.length,
  min: Math.min(...raw).toFixed(2) + '\u00B0C',
  max: Math.max(...raw).toFixed(2) + '\u00B0C',
  mean: (raw.reduce((a, b) => a + b) / raw.length).toFixed(2) + '\u00B0C',
};
summary

The annual averages tell the story at a glance — a gradual rise through the 2010s, then a sharp jump in 2023–2024 as anomalies cross the +1.0 °C threshold and keep climbing:

| Year | Mean (°C) | Peak (°C) | |
|------|-----------|-----------|---|
| 2010 | 0.72 | 0.92 | |
| 2011 | 0.61 | 0.75 | |
| 2012 | 0.65 | 0.80 | |
| 2013 | 0.68 | 0.83 | |
| 2014 | 0.75 | 0.88 | |
| 2015 | 0.90 | 1.17 | first peaks above +1 °C |
| 2016 | **1.01** | **1.37** | El Niño year |
| 2017 | 0.92 | 1.17 | |
| 2018 | 0.85 | 1.02 | |
| 2019 | 0.98 | 1.17 | |
| 2020 | **1.01** | 1.24 | tied warmest mean |
| 2021 | 0.85 | 0.99 | |
| 2022 | 0.89 | 1.05 | |
| 2023 | **1.17** | **1.48** | record shattered |
| 2024 | **1.28** | **1.45** | new record |

## Color Science with chroma-js

Mapping temperature to color sounds simple, but naive RGB interpolation produces muddy intermediate tones. The [chroma-js](https://gka.github.io/chroma.js/) library — imported directly from npm — gives us perceptually uniform interpolation in the CIELAB color space. A scale from deep blue through white to warm red mirrors the intuitive association between temperature and color, with no perceptual dead zones in between.

In [ ]:
const minAnom = 0.40;
const maxAnom = 1.50;

const tempScale = chroma
  .scale(['#2166ac', '#67a9cf', '#d1e5f0', '#fddbc7', '#ef8a62', '#b2182b'])
  .domain([minAnom, 0.65, 0.85, 1.00, 1.15, maxAnom])
  .mode('lab');

const thresholds = [0.5, 1.0, 1.5];

const params = {
  lineWeight: 2,
  glowLayers: 3,
  glowSpread: 4,
  dotSize: 7,
  bgColor: [8, 12, 24],
  labelColor: [180, 190, 210],
};

The scale is anchored at six key temperatures. Between these stops, chroma-js interpolates through CIELAB space — producing smooth transitions that our eyes perceive as even gradients, unlike the uneven jumps that RGB blending would create:

In [ ]:
[
  { anomaly: '0.40\u00B0C', hex: '#2166ac', note: 'coolest in dataset' },
  { anomaly: '0.65\u00B0C', hex: '#67a9cf' },
  { anomaly: '0.85\u00B0C', hex: '#d1e5f0' },
  { anomaly: '1.00\u00B0C', hex: '#fddbc7', note: 'crossing +1\u00B0C' },
  { anomaly: '1.15\u00B0C', hex: '#ef8a62' },
  { anomaly: '1.50\u00B0C', hex: '#b2182b', note: 'warmest in dataset' },
]

## The Spiral

Each data point maps to a position in polar coordinates: the **month** determines the angle (January at the top, progressing clockwise), and the **temperature anomaly** determines the distance from the center. As the animation progresses year by year, the line traces a spiral that visibly expands outward — especially in the final years, where anomalies routinely exceed +1.0 °C.

Once the spiral completes, a 3D rotation tilts the view 90° to reveal a flat side profile — each year becomes a horizontal band whose width shows how warm it was, with vertical reference lines marking the temperature thresholds. Use the controls at the bottom to pause, scrub through the timeline, or adjust the playback speed.

In [ ]:
let cx = 0, cy = 0, maxR = 1, minR = 0;
let currentIndex = 0;
let playing = true;
let timeline, speedSlider, playBtn;

// 3D cylinder transition
let tiltAngle = 0;
let targetTilt = HALF_PI; // 90° — ends as flat 2D side view
let yearSpacing = 0;
let transitionSteps = 180;
let pauseCountdown = 0;

function easeInOutCubic(t) {
  return t < 0.5 ? 4 * t * t * t : 1 - pow(-2 * t + 2, 3) / 2;
}

function polarXY(absMonth, anomaly) {
  let angle = (absMonth / 12) * TWO_PI - HALF_PI;
  let r = map(anomaly, minAnom, maxAnom, minR, maxR);
  let x = cos(angle) * r;
  let y = sin(angle) * r;

  if (tiltAngle > 0.001) {
    let yearIdx = floor(absMonth / 12);
    let z = (yearIdx - (totalYears - 1) / 2) * yearSpacing;
    return {
      x: cx + x,
      y: cy + y * cos(tiltAngle) - z * sin(tiltAngle)
    };
  }

  return { x: cx + x, y: cy + y };
}

function drawReferenceCircles() {
  let fade = tiltAngle > 0.001 ? max(0, 1 - tiltAngle / (targetTilt * 0.3)) : 1;
  if (fade < 0.01) return;

  noFill();
  for (let t of thresholds) {
    let r = map(t, minAnom, maxAnom, minR, maxR);
    let c = tempScale(t).rgb();
    stroke(c[0], c[1], c[2], 25 * fade);
    strokeWeight(0.5);
    circle(cx, cy, r * 2);
    fill(params.labelColor[0], params.labelColor[1], params.labelColor[2], 60 * fade);
    noStroke();
    textSize(11);
    text(t.toFixed(1) + '\u00B0C', cx + r + 6, cy - 4);
  }
}

function drawMonthLabels() {
  // fade out as rings compress toward the side view
  let monthFade = tiltAngle > 0.001 ? max(0, 1 - tiltAngle / (targetTilt * 0.55)) : 1;
  if (monthFade < 0.01) return;

  let labelR = maxR + 28;
  fill(params.labelColor[0], params.labelColor[1], params.labelColor[2], 80 * monthFade);
  noStroke();
  textSize(12);
  textAlign(CENTER, CENTER);

  let blend = constrain(tiltAngle / targetTilt, 0, 1);
  let zBottom = ((totalYears - 1) / 2) * yearSpacing;

  for (let m = 0; m < 12; m++) {
    let angle = (m / 12) * TWO_PI - HALF_PI;
    let lx = cx + cos(angle) * labelR;
    let lyFlat = cy + sin(angle) * labelR;
    let lyTilt = cy + sin(angle) * labelR * cos(tiltAngle) + zBottom * sin(tiltAngle) + 18;
    text(monthNames[m], lx, lerp(lyFlat, lyTilt, blend));
  }
}

## The Cylinder

Once the spiral completes, the view rotates a full 90° around the horizontal axis. The rings separate vertically — oldest at the bottom, newest at the top — passing through a 3D cylinder perspective before settling into a flat 2D side profile. In this final view each year is a horizontal band: wider bands mean warmer years, and vertical reference lines mark the temperature thresholds for easy comparison.

In [ ]:
function drawCylinderScale() {
  if (tiltAngle < 0.1) return;

  let fade = map(tiltAngle, 0.1, targetTilt * 0.5, 0, 1, true);
  if (fade < 0.01) return;

  let halfSpan = ((totalYears - 1) / 2) * yearSpacing;
  let topY = cy - halfSpan * sin(tiltAngle);
  let botY = cy + halfSpan * sin(tiltAngle);
  let pad = 24;

  for (let t of thresholds) {
    let r = map(t, minAnom, maxAnom, minR, maxR);
    let c = tempScale(t).rgb();

    stroke(c[0], c[1], c[2], 45 * fade);
    strokeWeight(0.8);
    line(cx - r, topY - pad, cx - r, botY + pad);
    line(cx + r, topY - pad, cx + r, botY + pad);

    noStroke();
    fill(c[0], c[1], c[2], 150 * fade);
    textSize(10);
    textAlign(CENTER, BOTTOM);
    text(t.toFixed(1) + '\u00B0C', cx - r, topY - pad - 2);
    text(t.toFixed(1) + '\u00B0C', cx + r, topY - pad - 2);
  }

  textAlign(CENTER, CENTER);
}

function drawCylinderLabels() {
  if (tiltAngle < 0.1) return;

  let fade = map(tiltAngle, 0.1, targetTilt * 0.5, 0, 200, true);
  textSize(11);
  noStroke();
  textAlign(LEFT, CENTER);

  for (let y = 0; y < totalYears; y++) {
    let z = (y - (totalYears - 1) / 2) * yearSpacing;
    let yPos = cy - z * sin(tiltAngle);
    let yearStart = y * 12;
    let yearEnd = min(yearStart + 12, data.length);
    let sum = 0;
    for (let i = yearStart; i < yearEnd; i++) sum += data[i].anomaly;
    let avgAnom = sum / (yearEnd - yearStart);
    let c = tempScale(avgAnom).rgb();
    fill(c[0], c[1], c[2], fade);
    text(startYear + y, cx + maxR + 15, yPos);
  }

  textAlign(CENTER, CENTER);
}

## Putting It Together

The sketch below ties everything together into a single animated canvas. It plays in three acts: first the spiral builds month by month, then a brief pause lets the completed shape register, and finally the view rotates 90° into a flat side profile. A control bar at the bottom provides play/pause, a timeline scrubber that covers both the spiral and the rotation, and a speed dial.

In [ ]:
function setup() {
  createCanvas(innerWidth, innerHeight);
  textFont('monospace');
  textAlign(CENTER, CENTER);

  cx = width / 2;
  cy = height / 2;
  maxR = min(width, height) * 0.38;
  minR = maxR * 0.18;
  yearSpacing = min(width, height) * 0.035;

  let totalFrames = data.length + transitionSteps;

  // control bar overlay
  let bar = createDiv('');
  bar.position(0, height - 48);
  bar.style('width', '100%');
  bar.style('display', 'flex');
  bar.style('align-items', 'center');
  bar.style('gap', '10px');
  bar.style('padding', '10px 20px');
  bar.style('background', 'rgba(8, 12, 24, 0.92)');
  bar.style('backdrop-filter', 'blur(8px)');
  bar.style('box-sizing', 'border-box');
  bar.style('font-family', 'monospace');
  bar.style('color', '#b4bed2');
  bar.style('font-size', '13px');
  bar.style('border-top', '1px solid rgba(180,190,210,0.1)');

  // play / pause
  playBtn = createButton('\u23F8');
  playBtn.parent(bar);
  playBtn.style('background', 'none');
  playBtn.style('border', '1px solid rgba(180,190,210,0.3)');
  playBtn.style('color', '#b4bed2');
  playBtn.style('font-size', '16px');
  playBtn.style('cursor', 'pointer');
  playBtn.style('padding', '2px 10px');
  playBtn.style('border-radius', '4px');
  playBtn.mousePressed(() => {
    if (currentIndex >= totalFrames) {
      currentIndex = 0;
      playing = true;
      pauseCountdown = 0;
    } else {
      playing = !playing;
    }
    playBtn.html(playing ? '\u23F8' : '\u25B6');
  });

  // timeline scrubber — covers spiral + 3D transition
  timeline = createSlider(0, totalFrames, 0, 1);
  timeline.parent(bar);
  timeline.style('flex', '1');
  timeline.style('accent-color', '#ef8a62');
  timeline.input(() => {
    playing = false;
    playBtn.html('\u25B6');
    currentIndex = timeline.value();
    pauseCountdown = 0;
  });

  // speed control
  let speedLabel = createSpan('Speed');
  speedLabel.parent(bar);
  speedSlider = createSlider(0.02, 0.3, 0.15, 0.01);
  speedSlider.parent(bar);
  speedSlider.style('width', '80px');
  speedSlider.style('accent-color', '#67a9cf');
}

function draw() {
  if (width === 0 || !speedSlider) return;

  background(...params.bgColor);

  let totalFrames = data.length + transitionSteps;

  // advance animation
  if (playing) {
    if (pauseCountdown > 0) {
      pauseCountdown--;
    } else if (currentIndex < data.length) {
      let prev = currentIndex;
      currentIndex = min(currentIndex + speedSlider.value(), data.length);
      if (prev < data.length && currentIndex >= data.length) {
        pauseCountdown = 60;
      }
    } else if (currentIndex < totalFrames) {
      currentIndex = min(currentIndex + 1, totalFrames);
    }
    timeline.value(floor(currentIndex));

    if (currentIndex >= totalFrames) {
      playing = false;
      playBtn.html('\u25B6');
    }
  }

  // compute tilt from slider position
  if (currentIndex > data.length) {
    let progress = constrain((currentIndex - data.length) / transitionSteps, 0, 1);
    tiltAngle = targetTilt * easeInOutCubic(progress);
  } else {
    tiltAngle = 0;
  }

  drawReferenceCircles();
  drawMonthLabels();

  let limit = floor(min(currentIndex, data.length));

  // draw the spiral
  noFill();
  for (let i = 1; i < limit; i++) {
    let d0 = data[i - 1];
    let d1 = data[i];
    let p0 = polarXY(d0.month + (d0.year - startYear) * 12, d0.anomaly);
    let p1 = polarXY(d1.month + (d1.year - startYear) * 12, d1.anomaly);
    let c = tempScale(d1.anomaly).rgb();

    for (let g = params.glowLayers; g >= 0; g--) {
      let alpha = g === 0 ? 220 : 30 / g;
      let weight = g === 0 ? params.lineWeight : params.lineWeight + g * params.glowSpread;
      stroke(c[0], c[1], c[2], alpha);
      strokeWeight(weight);
      line(p0.x, p0.y, p1.x, p1.y);
    }
  }

  // drawing head (fades during transition)
  if (limit > 0 && limit <= data.length) {
    let headFade = tiltAngle > 0.001 ? max(0, 1 - tiltAngle / (targetTilt * 0.2)) : 1;
    if (headFade > 0.01) {
      let d = data[min(limit - 1, data.length - 1)];
      let p = polarXY(d.month + (d.year - startYear) * 12, d.anomaly);
      let c = tempScale(d.anomaly).rgb();
      noStroke();
      fill(c[0], c[1], c[2], 40 * headFade);
      circle(p.x, p.y, params.dotSize * 4);
      fill(c[0], c[1], c[2], 120 * headFade);
      circle(p.x, p.y, params.dotSize * 2);
      fill(255, 255, 255, 230 * headFade);
      circle(p.x, p.y, params.dotSize * 0.6);
    }
  }

  // cylinder overlay: scale lines + year labels
  drawCylinderScale();
  drawCylinderLabels();

  // center label (fades during transition)
  let centerFade = tiltAngle > 0.001 ? max(0, 1 - tiltAngle / (targetTilt * 0.3)) : 1;
  if (limit > 0 && centerFade > 0.01) {
    let d = data[min(limit - 1, data.length - 1)];
    let c = tempScale(d.anomaly).rgb();
    fill(c[0], c[1], c[2], 200 * centerFade);
    noStroke();
    textSize(28);
    text(d.year, cx, cy - 8);
    fill(params.labelColor[0], params.labelColor[1], params.labelColor[2], 80 * centerFade);
    textSize(13);
    text(monthNames[d.month], cx, cy + 16);
    fill(params.labelColor[0], params.labelColor[1], params.labelColor[2], 40 * centerFade);
    textSize(11);
    text('global temperature anomaly', cx, cy + 34);
  }

  // title in cylinder view
  if (tiltAngle > targetTilt * 0.3) {
    let titleAlpha = map(tiltAngle, targetTilt * 0.3, targetTilt * 0.7, 0, 200, true);
    fill(params.labelColor[0], params.labelColor[1], params.labelColor[2], titleAlpha);
    noStroke();
    textSize(20);
    text('Global Temperature Anomaly', cx, 50);
    textSize(14);
    fill(params.labelColor[0], params.labelColor[1], params.labelColor[2], titleAlpha * 0.6);
    text(startYear + '\u2013' + (startYear + totalYears - 1), cx, 74);
  }

  // attribution
  fill(params.labelColor[0], params.labelColor[1], params.labelColor[2], currentIndex >= data.length ? 70 : 50);
  textSize(14);
  textAlign(LEFT, TOP);
  text('NASA GISS  \u00B7  \u00B0C vs 1951\u20131980 baseline', 20, 20);
  textAlign(CENTER, CENTER);
}

In [ ]:
%show 100% 800px